# Local neurotransmitter fingerprinting

In this tutorial, you will explore the local neurotransmitter fingerprinting (LNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch and prepare the neurotransmitter PET atlas
- Compute per-target neurotransmitter density scores within a lesion
- Filter by neurotransmitter system presets
- Obtain parcel-level NT scores

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/06-local-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

Get the tutorial data.

In [1]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force


Setting up tutorial data at: /tmp/tutorial_bids
✓ Tutorial data copied to: /tmp/tutorial_bids

The tutorial dataset includes:
  - 3 synthetic subjects (sub-01, sub-02, sub-03)
  - Binary lesion masks in MNI152NLin6Asym space
  - BIDS-style structure


## Fetch neurotransmitter atlas

Local neurotransmitter fingerprinting requires a neurotransmitter PET atlas: a collection of PET receptor/transporter density maps from normative cohorts. Lacuna downloads these from the [Open Science Framework](https://osf.io/yz9mb/).

The atlas includes maps for multiple neurotransmitter targets (e.g., D1, 5HT1a, DAT, GABAa) derived from published PET tracer studies.

In [2]:
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

Fetching neurotransmitter PET atlas maps from OSF...
  Output: /tmp/ntatlas_data

[1/45] target-MOR_tracer-carfentanil_n-204_dx-hc_pub-kantonen2020_space-MNI152NL
[2/45] target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017_space-MNI152NLin6Asym_desc
[3/45] target-5HT1a_tracer-cumi101_n-8_dx-hc_pub-beliveau2017_space-MNI152NLin6A
[4/45] target-GABAa_tracer-flumazenil_n-10_dx-hc_pub-kaulen2022_space-MNI152NLin
[5/45] target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017_space-MNI152NLin6Asym
[6/45] target-M1_tracer-lsn3172176_n-24_dx-hc_pub-naganawa2020_space-MNI152NLin6
[7/45] target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019_space-MNI152NLin6Asy
[8/45] target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015_space-MNI152NLin6Asy
[9/45] target-DAT_tracer-fepe2i_n-6_dx-hc_pub-sasaki2012_space-MNI152NLin6Asym_d
[10/45] target-5HT2a_tracer-cimbi36_n-29_dx-hc_pub-beliveau2017_space-MNI152NLin
[11/45] target-GABAa_tracer-flumazenil_n-16_dx-hc_pub-norgaard2021_space-MNI152N
[12/45] target-5HT2a_tracer

## Prepare the atlas

Before running the analysis, the raw PET maps need to be prepared. This step:
1. Groups maps by neurotransmitter target
2. Averages maps within each target (excluding zeros)
3. Z-scores the result
4. Caches the processed atlas for reuse

In [8]:
!lacuna prepare lntf --help

usage: lacuna prepare lntf [-h] [--source-dir SOURCE_DIR]
                           [--cache-dir CACHE_DIR] [--map-config MAP_CONFIG]

options:
  -h, --help            show this help message and exit
  --source-dir SOURCE_DIR
                        Directory with raw PET NIfTI maps
  --cache-dir CACHE_DIR
                        Output cache directory
  --map-config MAP_CONFIG
                        YAML map selection config file


In [9]:
!lacuna prepare lntf \
    --source-dir /tmp/ntatlas_data \
    --cache-dir /tmp/ntatlas_cache

2026-04-21 08:26:52 - lacuna.cli.prepare - INFO - Building NT atlas from /tmp/ntatlas_data
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HT1a from 2 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HT1b from 3 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HT2a from 2 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HT4 from 1 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HT6 from 1 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building 5HTT from 3 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building A4B2 from 1 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building CB1 from 2 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building D1 from 1 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building D23 from 5 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building DAT from 3 maps
2026-04-21 08:26:52 - lacuna.atlas.store - INFO - Building FDOPA from 1 map

## Analysis

Local neurotransmitter fingerprinting scores the z-scored PET atlas values directly within the lesion mask. For each neurotransmitter target, it computes the mean (or sum) of the atlas values at lesion voxels.

This answers: **what neurotransmitter landscape did the lesion wipe out?**

A high score for a given target indicates that the lesioned region is rich in that neurotransmitter system, suggesting potential neurochemical consequences of the lesion.

Run the analysis.

In [ ]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

In [6]:
!lacuna run lntf --help

usage: lacuna run lntf [-h] [--participant-label LABEL [LABEL ...]]
                       [--session-id SESSION [SESSION ...]] [--pattern GLOB]
                       [--mask-space SPACE] [--overwrite]
                       [--on-empty {warn,skip,error}] [--keep-intermediate]
                       [-v] [--targets TARGETS] [--enriched]
                       [--ace-cache-dir ACE_CACHE_DIR]
                       [--atlas-cache-dir ATLAS_CACHE_DIR]
                       [--aggregation {mean,sum}]
                       bids_dir output_dir

Local neurotransmitter fingerprinting: score NT atlas values directly
within the lesion mask. Answers: 'what neurotransmitter landscape
did the lesion wipe out?'

Examples:
  lacuna run lntf /bids /output
  lacuna run lntf /bids /output --targets dopaminergic
  lacuna run lntf /bids /output --parcel-atlases schaefer2018parcels100networks7

positional arguments:
  bids_dir              Root folder of BIDS dataset (sub-XXXXX folders at top
          

List the outputs.

In [ ]:
!ls /tmp/outputs_lntf/sub-01/ses-01/anat/

Visualize the per-target neurotransmitter scores.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load the LNTF scores from the JSON sidecar
results_dir = Path("/tmp/outputs_lntf/sub-01/ses-01/anat/")
score_files = sorted(results_dir.glob("*method-lntf*scores*.json"))

# If scores are in TSV format instead
import pandas as pd
tsv_files = sorted(results_dir.glob("*method-lntf*parcelstats.tsv"))

if tsv_files:
    df = pd.read_csv(tsv_files[0], sep="\t")
    print(df)

## Filter by neurotransmitter system

Instead of computing all targets, you can restrict the analysis to a specific neurotransmitter system using the `--targets` flag with a preset name:

| Preset | Targets |
|--------|--------|
| `dopaminergic` | D1, D23, DAT, FDOPA |
| `serotonergic` | 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT |
| `cholinergic` | VAChT, M1, A4B2 |
| `monoaminergic` | D1, D23, DAT, 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT, NET |
| `all` | All available targets (default) |

You can also pass a comma-separated list of individual targets (e.g., `--targets D1,DAT,5HT2a`).

In [ ]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf_dopamine/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache \
    --targets dopaminergic

## Obtain parcel-level NT scores

Beyond global lesion-level scores, Lacuna can also compute NT scores per atlas parcel. This provides a spatial profile of the neurotransmitter landscape across brain regions.

In [ ]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf_parcels/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache \
    --parcel-atlases schaefer2018parcels100networks7 \
    --verbose

In [ ]:
!ls /tmp/outputs_lntf_parcels/sub-01/ses-01/anat/

## Run on multiple subjects

Lacuna supports processing multiple subjects within a single run. If the `--participant-label` flag is omitted, the pipeline automatically processes all subjects detected in the BIDS dataset.

LNTF is very fast since it only requires voxel lookups in the atlas — no connectome loading is needed.

In [ ]:
!lacuna run lntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_lntf_all/ \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

Collect results into a group-level table.

In [ ]:
!lacuna collect \
    /tmp/outputs_lntf_all/ \
    --pattern "*lntf*parcelstats*" \
    --output-dir /tmp/group_lntm/

In [ ]:
!ls /tmp/group_lntm/